# T-Learner Uplift Models

This notebook implements logistic-regression and XGBoost T-learners for the binary Hillstrom email treatment.

Each T-learner fits two separate outcome models:

- **Treatment model:** estimates the probability of a visit if a customer receives an email.
- **Control model:** estimates the probability of a visit if a customer receives no email.

Estimated uplift is:

$$
\hat{\tau}(X)
=
\hat{P}(Y=1 \mid X,T=1)
-
\hat{P}(Y=1 \mid X,T=0)
$$

The treatment combines the men's and women's email groups into one email-treatment group.

### Import required libraries

This cell imports the data-processing, modeling, visualization, and uplift-evaluation tools used throughout the notebook.

In [1]:
import sys
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    brier_score_loss,
)
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

from sklearn.model_selection import GridSearchCV, StratifiedKFold

from causalml.metrics import (
    plot_qini,
    plot_gain,
    qini_score,
    auuc_score,
)

pd.set_option("display.max_columns", 100)
plt.style.use("seaborn-v0_8-darkgrid")

### Configure project paths

This cell locates the repository root, imports shared project settings, and ensures the T-learner output directories exist.

In [2]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    PROCESSED_DATA_DIR,
    PREDICTIONS_DIR,
    MODEL_RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    RANDOM_STATE,
    PRIMARY_OUTCOME,
)

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DATA_DIR)
print("Predictions output:", PREDICTIONS_DIR)
print("Results output:", MODEL_RESULTS_DIR)

from src.metrics import compute_qini_auuc, stratified_uplift_folds
from src.plotting import (
    plot_cumulative_gain_curve,
    plot_qini_curve,
    save_fig,
)


Project root: /workspace/scratch/61d1d4e14f7f/hillstrom-email-uplift-modeling
Processed data: /workspace/scratch/61d1d4e14f7f/hillstrom-email-uplift-modeling/data/processed
Predictions output: /workspace/scratch/61d1d4e14f7f/hillstrom-email-uplift-modeling/outputs/predictions
Results output: /workspace/scratch/61d1d4e14f7f/hillstrom-email-uplift-modeling/outputs/model_results


### Load processed modeling data

This cell loads the shared feature matrices, visit outcomes, binary treatment indicators, and original campaign labels.

In [3]:
X_train = pd.read_csv(PROCESSED_DATA_DIR / "X_train.csv")
X_test = pd.read_csv(PROCESSED_DATA_DIR / "X_test.csv")

y_train = pd.read_csv(
    PROCESSED_DATA_DIR / "y_train.csv"
).squeeze("columns")

y_test = pd.read_csv(
    PROCESSED_DATA_DIR / "y_test.csv"
).squeeze("columns")

treatment_train = pd.read_csv(
    PROCESSED_DATA_DIR / "treatment_train.csv"
).squeeze("columns")

treatment_test = pd.read_csv(
    PROCESSED_DATA_DIR / "treatment_test.csv"
).squeeze("columns")

segment_test = pd.read_csv(
    PROCESSED_DATA_DIR / "segment_test.csv"
).squeeze("columns")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nTraining treatment counts:")
print(treatment_train.value_counts().sort_index())

print("\nTesting treatment counts:")
print(treatment_test.value_counts().sort_index())

print("\nTraining visit rates:")
print(
    pd.DataFrame({
        "visit": y_train,
        "treatment": treatment_train
    })
    .groupby("treatment")["visit"]
    .mean()
)

X_train shape: (48000, 18)
X_test shape: (16000, 18)

Training treatment counts:
treatment_binary
0    15980
1    32020
Name: count, dtype: int64

Testing treatment counts:
treatment_binary
0     5326
1    10674
Name: count, dtype: int64

Training visit rates:
treatment
0    0.106195
1    0.167021
Name: visit, dtype: float64


### Validate the modeling data

This cell confirms that the feature, outcome, and treatment files have consistent dimensions and valid binary values.

In [4]:
assert list(X_train.columns) == list(X_test.columns)

assert len(X_train) == len(y_train) == len(treatment_train)
assert len(X_test) == len(y_test) == len(treatment_test)

assert set(y_train.unique()).issubset({0, 1})
assert set(y_test.unique()).issubset({0, 1})

assert set(treatment_train.unique()).issubset({0, 1})
assert set(treatment_test.unique()).issubset({0, 1})

assert X_train.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0

print("Data validation passed.")

Data validation passed.


### Separate treated and control observations

This cell creates the treatment and control training samples that both T-learners will use to fit their two outcome models.

In [5]:
train_treated_mask = treatment_train == 1
train_control_mask = treatment_train == 0

test_treated_mask = treatment_test == 1
test_control_mask = treatment_test == 0

X_train_treated = X_train.loc[train_treated_mask]
y_train_treated = y_train.loc[train_treated_mask]

X_train_control = X_train.loc[train_control_mask]
y_train_control = y_train.loc[train_control_mask]

print("Treated training observations:", len(X_train_treated))
print("Control training observations:", len(X_train_control))

print("\nTraining visit rates:")
print("Treated:", y_train_treated.mean())
print("Control:", y_train_control.mean())

assert y_train_treated.nunique() == 2
assert y_train_control.nunique() == 2

Treated training observations: 32020
Control training observations: 15980

Training visit rates:
Treated: 0.1670206121174266
Control: 0.10619524405506883


### Cross-validation Helper

In [6]:
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)


def fit_with_cv(estimator, parameter_grid, X, y):
    search = GridSearchCV(
        estimator=estimator,
        param_grid=parameter_grid,
        scoring="neg_log_loss",
        cv=cv_strategy,
        refit=True,
        n_jobs=1,
    )

    search.fit(X, y)

    return search.best_estimator_, search

### Train the logistic-regression T-learner

This cell fits separate logistic-regression outcome models using the treated and control training observations.

In [7]:
lgr_parameters = {
    "max_iter": 2000,
    "solver": "lbfgs",
}

lgr_parameter_grid = {
    "C": [0.1, 1.0, 10.0],
}

lgr_treated_model, lgr_treated_cv = fit_with_cv(
    LogisticRegression(**lgr_parameters),
    lgr_parameter_grid,
    X_train_treated,
    y_train_treated,
)

lgr_control_model, lgr_control_cv = fit_with_cv(
    LogisticRegression(**lgr_parameters),
    lgr_parameter_grid,
    X_train_control,
    y_train_control,
)

print("Treated best parameters:", lgr_treated_cv.best_params_)
print("Control best parameters:", lgr_control_cv.best_params_)

Treated best parameters: {'C': 1.0}
Control best parameters: {'C': 10.0}


### Train the XGBoost T-learner

This cell fits separate XGBoost outcome models that can capture nonlinear customer patterns and feature interactions.

In [8]:
xgb_parameters = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "n_estimators": 200,
    "max_depth": 3,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": RANDOM_STATE,
    "n_jobs": 1,
}

xgb_parameter_grid = {
    "max_depth": [2, 3],
    "learning_rate": [0.05, 0.1],
}

xgb_treated_model, xgb_treated_cv = fit_with_cv(
    xgb.XGBClassifier(**xgb_parameters),
    xgb_parameter_grid,
    X_train_treated,
    y_train_treated,
)

xgb_control_model, xgb_control_cv = fit_with_cv(
    xgb.XGBClassifier(**xgb_parameters),
    xgb_parameter_grid,
    X_train_control,
    y_train_control,
)

print("Treated best parameters:", xgb_treated_cv.best_params_)
print("Control best parameters:", xgb_control_cv.best_params_)

Treated best parameters: {'learning_rate': 0.05, 'max_depth': 2}
Control best parameters: {'learning_rate': 0.05, 'max_depth': 2}


### Train the Random Forest T-learner

This cell fits separate regularized random forests to estimate visit probabilities under treatment and control.

In [9]:
rf_parameters = {
    "n_estimators": 300,
    "criterion": "gini",
    "max_depth": 8,
    "min_samples_leaf": 50,
    "max_features": "sqrt",
    "bootstrap": True,
    "random_state": RANDOM_STATE,
    "n_jobs": 1,
}

rf_parameter_grid = {
    "max_depth": [6, 8],
    "min_samples_leaf": [25, 50],
}

rf_treated_model, rf_treated_cv = fit_with_cv(
    RandomForestClassifier(**rf_parameters),
    rf_parameter_grid,
    X_train_treated,
    y_train_treated,
)

rf_control_model, rf_control_cv = fit_with_cv(
    RandomForestClassifier(**rf_parameters),
    rf_parameter_grid,
    X_train_control,
    y_train_control,
)

print("Treated best parameters:", rf_treated_cv.best_params_)
print("Control best parameters:", rf_control_cv.best_params_)

Treated best parameters: {'max_depth': 8, 'min_samples_leaf': 50}
Control best parameters: {'max_depth': 8, 'min_samples_leaf': 50}


### Training-only out-of-fold uplift predictions

Each validation customer is scored by treated and control models fitted without that customer. The fixed within-family configurations were selected using outcome-model CV on the training split; family selection uses these OOF uplift metrics and never test Qini.


In [10]:
N_OOF_FOLDS = 5
t_model_templates = {
    "T-Learner (LR)": (lgr_treated_model, lgr_control_model),
    "T-Learner (XGB)": (xgb_treated_model, xgb_control_model),
    "T-Learner (RF)": (rf_treated_model, rf_control_model),
}
t_oof = {name: np.full(len(X_train), np.nan) for name in t_model_templates}

for fold_train_idx, fold_val_idx in stratified_uplift_folds(
    y_train, treatment_train, n_splits=N_OOF_FOLDS, random_state=RANDOM_STATE
):
    fold_treatment = treatment_train.iloc[fold_train_idx].to_numpy()
    for name, (treated_template, control_template) in t_model_templates.items():
        treated_model = clone(treated_template)
        control_model = clone(control_template)
        treated_rows = fold_train_idx[fold_treatment == 1]
        control_rows = fold_train_idx[fold_treatment == 0]
        treated_model.fit(X_train.iloc[treated_rows], y_train.iloc[treated_rows])
        control_model.fit(X_train.iloc[control_rows], y_train.iloc[control_rows])
        t_oof[name][fold_val_idx] = (
            treated_model.predict_proba(X_train.iloc[fold_val_idx])[:, 1]
            - control_model.predict_proba(X_train.iloc[fold_val_idx])[:, 1]
        )

assert all(np.isfinite(values).all() for values in t_oof.values())
t_validation_scores = compute_qini_auuc(
    y_train, treatment_train, t_oof, normalize=True
)
t_validation_comparison = pd.DataFrame({
    "qini_score": t_validation_scores["qini_score"],
    "auuc_score": t_validation_scores["auuc_score"],
}).sort_values("qini_score", ascending=False)
display(t_validation_comparison)


,qini_score,auuc_score
T-Learner (XGB),0.080797,0.579523
T-Learner (LR),0.062136,0.561356
T-Learner (RF),0.057276,0.555275


### Estimate uplift with logistic regression

This cell scores every test customer under both treatment conditions and subtracts the control probability from the treatment probability.

In [11]:
lgr_pred_visit_if_treated = (
    lgr_treated_model.predict_proba(X_test)[:, 1]
)

lgr_pred_visit_if_control = (
    lgr_control_model.predict_proba(X_test)[:, 1]
)

lgr_predicted_uplift = (
    lgr_pred_visit_if_treated
    - lgr_pred_visit_if_control
)

### Estimate uplift with XGBoost

This cell uses the two XGBoost outcome models to calculate an alternative uplift estimate for every test customer.

In [12]:
xgb_pred_visit_if_treated = (
    xgb_treated_model.predict_proba(X_test)[:, 1]
)

xgb_pred_visit_if_control = (
    xgb_control_model.predict_proba(X_test)[:, 1]
)

xgb_predicted_uplift = (
    xgb_pred_visit_if_treated
    - xgb_pred_visit_if_control
)

### Estimate uplift with Random Forest

This cell calculates treatment and control visit probabilities from the two forests and subtracts them to estimate uplift.

In [13]:
rf_pred_visit_if_treated = (
    rf_treated_model.predict_proba(X_test)[:, 1]
)

rf_pred_visit_if_control = (
    rf_control_model.predict_proba(X_test)[:, 1]
)

rf_predicted_uplift = (
    rf_pred_visit_if_treated
    - rf_pred_visit_if_control
)

### Calculate the observed test-set ATE

This cell calculates the randomized test-set difference in visit rates between customers who received an email and those who did not.

In [14]:
observed_test_ate = (
    y_test.loc[test_treated_mask].mean()
    - y_test.loc[test_control_mask].mean()
)

print(f"Observed test-set ATE: {observed_test_ate:.4f}")

Observed test-set ATE: 0.0611


### Summarize logistic-regression uplift

This cell compares the logistic model's average predicted uplift with the observed ATE and reports its predicted-uplift range.

In [15]:
lgr_average_predicted_uplift = (
    lgr_predicted_uplift.mean()
)

lgr_minimum_predicted_uplift = (
    lgr_predicted_uplift.min()
)

lgr_maximum_predicted_uplift = (
    lgr_predicted_uplift.max()
)

print("Logistic-Regression T-Learner")
print("=" * 45)
print(f"Observed test-set ATE:       {observed_test_ate:.4f}")
print(f"Average predicted uplift:   {lgr_average_predicted_uplift:.4f}")
print(f"Minimum predicted uplift:   {lgr_minimum_predicted_uplift:.4f}")
print(f"Maximum predicted uplift:   {lgr_maximum_predicted_uplift:.4f}")

Logistic-Regression T-Learner
Observed test-set ATE:       0.0611
Average predicted uplift:   0.0599
Minimum predicted uplift:   -0.0365
Maximum predicted uplift:   0.1223


### Summarize XGBoost uplift

This cell compares the XGBoost model's average predicted uplift with the same observed ATE and reports its predicted-uplift range.

In [16]:
xgb_average_predicted_uplift = (
    xgb_predicted_uplift.mean()
)

xgb_minimum_predicted_uplift = (
    xgb_predicted_uplift.min()
)

xgb_maximum_predicted_uplift = (
    xgb_predicted_uplift.max()
)

print("XGBoost T-Learner")
print("=" * 45)
print(f"Observed test-set ATE:       {observed_test_ate:.4f}")
print(f"Average predicted uplift:   {xgb_average_predicted_uplift:.4f}")
print(f"Minimum predicted uplift:   {xgb_minimum_predicted_uplift:.4f}")
print(f"Maximum predicted uplift:   {xgb_maximum_predicted_uplift:.4f}")

XGBoost T-Learner
Observed test-set ATE:       0.0611
Average predicted uplift:   0.0600
Minimum predicted uplift:   -0.0686
Maximum predicted uplift:   0.2071


### Summarize Random Forest uplift

This cell compares the Random Forest's average predicted uplift with the observed ATE and displays its predicted range.

In [17]:
rf_average_predicted_uplift = (
    rf_predicted_uplift.mean()
)

rf_minimum_predicted_uplift = (
    rf_predicted_uplift.min()
)

rf_maximum_predicted_uplift = (
    rf_predicted_uplift.max()
)

print("Random Forest T-Learner")
print("=" * 45)
print(f"Observed test-set ATE:       {observed_test_ate:.4f}")
print(f"Average predicted uplift:   {rf_average_predicted_uplift:.4f}")
print(f"Minimum predicted uplift:   {rf_minimum_predicted_uplift:.4f}")
print(f"Maximum predicted uplift:   {rf_maximum_predicted_uplift:.4f}")

Random Forest T-Learner
Observed test-set ATE:       0.0611
Average predicted uplift:   0.0599
Minimum predicted uplift:   -0.0293
Maximum predicted uplift:   0.1625


### Evaluate logistic-regression outcome models

This cell evaluates each logistic outcome model only on test customers whose corresponding treatment condition was actually observed.

In [18]:
lgr_treated_auc = roc_auc_score(
    y_test.loc[test_treated_mask],
    lgr_pred_visit_if_treated[test_treated_mask]
)

lgr_control_auc = roc_auc_score(
    y_test.loc[test_control_mask],
    lgr_pred_visit_if_control[test_control_mask]
)

lgr_treated_brier = brier_score_loss(
    y_test.loc[test_treated_mask],
    lgr_pred_visit_if_treated[test_treated_mask]
)

lgr_control_brier = brier_score_loss(
    y_test.loc[test_control_mask],
    lgr_pred_visit_if_control[test_control_mask]
)

print("Logistic-Regression Factual Performance")
print("=" * 45)
print(f"Treatment-model ROC AUC: {lgr_treated_auc:.4f}")
print(f"Control-model ROC AUC:   {lgr_control_auc:.4f}")
print(f"Treatment-model Brier:   {lgr_treated_brier:.4f}")
print(f"Control-model Brier:     {lgr_control_brier:.4f}")

Logistic-Regression Factual Performance
Treatment-model ROC AUC: 0.6231
Control-model ROC AUC:   0.6375
Treatment-model Brier:   0.1357
Control-model Brier:     0.0932


### Evaluate XGBoost outcome models

This cell evaluates the XGBoost treatment model on treated customers and the XGBoost control model on control customers.

In [19]:
xgb_treated_auc = roc_auc_score(
    y_test.loc[test_treated_mask],
    xgb_pred_visit_if_treated[test_treated_mask]
)

xgb_control_auc = roc_auc_score(
    y_test.loc[test_control_mask],
    xgb_pred_visit_if_control[test_control_mask]
)

xgb_treated_brier = brier_score_loss(
    y_test.loc[test_treated_mask],
    xgb_pred_visit_if_treated[test_treated_mask]
)

xgb_control_brier = brier_score_loss(
    y_test.loc[test_control_mask],
    xgb_pred_visit_if_control[test_control_mask]
)

print("XGBoost Factual Performance")
print("=" * 45)
print(f"Treatment-model ROC AUC: {xgb_treated_auc:.4f}")
print(f"Control-model ROC AUC:   {xgb_control_auc:.4f}")
print(f"Treatment-model Brier:   {xgb_treated_brier:.4f}")
print(f"Control-model Brier:     {xgb_control_brier:.4f}")

XGBoost Factual Performance
Treatment-model ROC AUC: 0.6201
Control-model ROC AUC:   0.6264
Treatment-model Brier:   0.1357
Control-model Brier:     0.0935


### Evaluate Random Forest outcome models

This cell evaluates the Random Forest treatment model on treated customers and the control model on control customers.

In [20]:
rf_treated_auc = roc_auc_score(
    y_test.loc[test_treated_mask],
    rf_pred_visit_if_treated[test_treated_mask]
)

rf_control_auc = roc_auc_score(
    y_test.loc[test_control_mask],
    rf_pred_visit_if_control[test_control_mask]
)

rf_treated_brier = brier_score_loss(
    y_test.loc[test_treated_mask],
    rf_pred_visit_if_treated[test_treated_mask]
)

rf_control_brier = brier_score_loss(
    y_test.loc[test_control_mask],
    rf_pred_visit_if_control[test_control_mask]
)

print("Random Forest Factual Performance")
print("=" * 45)
print(f"Treatment-model ROC AUC: {rf_treated_auc:.4f}")
print(f"Control-model ROC AUC:   {rf_control_auc:.4f}")
print(f"Treatment-model Brier:   {rf_treated_brier:.4f}")
print(f"Control-model Brier:     {rf_control_brier:.4f}")

Random Forest Factual Performance
Treatment-model ROC AUC: 0.6185
Control-model ROC AUC:   0.6147
Treatment-model Brier:   0.1359
Control-model Brier:     0.0935


### Compare factual classification performance

This cell displays the ROC AUC and Brier scores for all treatment-specific outcome models.

In [21]:
factual_comparison = pd.DataFrame({
    "Model": [
        "Logistic treatment model",
        "Logistic control model",
        "XGBoost treatment model",
        "XGBoost control model",
        "Random Forest treatment model",
        "Random Forest control model",
    ],
    "ROC AUC": [
        lgr_treated_auc,
        lgr_control_auc,
        xgb_treated_auc,
        xgb_control_auc,
        rf_treated_auc,
        rf_control_auc,
    ],
    "Brier Score": [
        lgr_treated_brier,
        lgr_control_brier,
        xgb_treated_brier,
        xgb_control_brier,
        rf_treated_brier,
        rf_control_brier,
    ],
})

display(factual_comparison)

,Model,ROC AUC,Brier Score
0,Logistic treatment model,0.623080,0.135651
1,Logistic control model,0.637533,0.093213
2,XGBoost treatment model,0.620073,0.135700
3,XGBoost control model,0.626371,0.093499
4,Random Forest treatment model,0.618527,0.135852
5,Random Forest control model,0.614728,0.093453


### Combine model predictions

This cell places the observed outcomes, treatment labels, potential-outcome predictions, and uplift estimates from all models into one comparison dataset.

In [22]:
model_comparison_df = pd.DataFrame({
    "test_row_id": np.arange(len(y_test)),
    "y_true": y_test.to_numpy(),
    "treatment": treatment_test.to_numpy(),
    "segment": segment_test.to_numpy(),
    "lgr_pred_visit_if_treated": lgr_pred_visit_if_treated,
    "lgr_pred_visit_if_control": lgr_pred_visit_if_control,
    "lgr_predicted_uplift": lgr_predicted_uplift,
    "xgb_pred_visit_if_treated": xgb_pred_visit_if_treated,
    "xgb_pred_visit_if_control": xgb_pred_visit_if_control,
    "xgb_predicted_uplift": xgb_predicted_uplift,
    "rf_pred_visit_if_treated": rf_pred_visit_if_treated,
    "rf_pred_visit_if_control": rf_pred_visit_if_control,
    "rf_predicted_uplift": rf_predicted_uplift,
})

display(model_comparison_df.head(10))

,test_row_id,y_true,treatment,segment,lgr_pred_visit_if_treated,lgr_pred_visit_if_control,lgr_predicted_uplift,xgb_pred_visit_if_treated,xgb_pred_visit_if_control,xgb_predicted_uplift,rf_pred_visit_if_treated,rf_pred_visit_if_control,rf_predicted_uplift
0,0,0,0,No E-Mail,0.100396,0.053354,0.047042,0.105212,0.056562,0.048650,0.102537,0.048884,0.053653
1,1,0,1,Womens E-Mail,0.210674,0.199938,0.010735,0.217832,0.169667,0.048165,0.209319,0.170513,0.038806
2,2,0,1,Mens E-Mail,0.143256,0.092746,0.050510,0.136935,0.100232,0.036702,0.153311,0.119666,0.033645
3,3,1,0,No E-Mail,0.160675,0.109463,0.051211,0.151853,0.110799,0.041054,0.141297,0.123809,0.017488
4,4,1,1,Mens E-Mail,0.192783,0.172841,0.019942,0.199306,0.146965,0.052342,0.198604,0.162747,0.035857
5,5,0,1,Mens E-Mail,0.310180,0.242946,0.067233,0.300915,0.200329,0.100587,0.240405,0.146415,0.093991
6,6,0,1,Womens E-Mail,0.207849,0.161802,0.046047,0.213393,0.181417,0.031976,0.221336,0.192958,0.028378
7,7,1,0,No E-Mail,0.099561,0.063359,0.036202,0.106238,0.073976,0.032261,0.113658,0.091217,0.022441
8,8,0,0,No E-Mail,0.178988,0.088528,0.090460,0.177724,0.088297,0.089426,0.183015,0.090346,0.092669
9,9,0,0,No E-Mail,0.234822,0.191757,0.043065,0.213008,0.161687,0.051321,0.216767,0.177681,0.039086


### Compare predicted-uplift distributions

This cell compares how widely logistic regression and XGBoost distribute their customer-level uplift estimates.

In [23]:
uplift_minimum = min(
    lgr_predicted_uplift.min(),
    xgb_predicted_uplift.min(),
    rf_predicted_uplift.min()
)

uplift_maximum = max(
    lgr_predicted_uplift.max(),
    xgb_predicted_uplift.max(),
    rf_predicted_uplift.max()
)

shared_bins = np.linspace(
    uplift_minimum,
    uplift_maximum,
    41
)

plt.figure(figsize=(10, 6))

plt.hist(
    lgr_predicted_uplift,
    bins=shared_bins,
    alpha=0.45,
    edgecolor="black",
    label="Logistic T-learner"
)

plt.hist(
    xgb_predicted_uplift,
    bins=shared_bins,
    alpha=0.45,
    edgecolor="black",
    label="XGBoost T-learner"
)

plt.hist(
    rf_predicted_uplift,
    bins=shared_bins,
    alpha=0.45,
    edgecolor="black",
    label="Random Forest T-learner"
)

plt.axvline(
    0,
    color="black",
    linestyle="--",
    label="Zero uplift"
)

plt.axvline(
    lgr_average_predicted_uplift,
    color="blue",
    linestyle="--",
    label=f"Logistic mean = {lgr_average_predicted_uplift:.3f}"
)

plt.axvline(
    xgb_average_predicted_uplift,
    color="red",
    linestyle="--",
    label=f"XGBoost mean = {xgb_average_predicted_uplift:.3f}"
)

plt.axvline(
    rf_average_predicted_uplift,
    color="green",
    linestyle="--",
    label=f"Random Forest mean = {rf_average_predicted_uplift:.3f}"
)

plt.xlabel("Predicted Uplift")
plt.ylabel("Number of Customers")
plt.title("Distribution of T-Learner Predicted Uplift")
plt.legend()
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "t_learner_uplift_distributions.png",
    dpi=150
)

plt.show()

### Define uplift-by-decile evaluation

This function ranks customers from highest to lowest predicted uplift and calculates observed treatment-control lift within each equally sized decile.

In [24]:
def calculate_uplift_by_decile(
    y_true,
    treatment,
    predicted_uplift,
    number_of_deciles=10
):
    decile_data = pd.DataFrame({
        "y_true": np.asarray(y_true),
        "treatment": np.asarray(treatment),
        "predicted_uplift": np.asarray(predicted_uplift),
    })

    decile_data = (
        decile_data
        .sort_values(
            "predicted_uplift",
            ascending=False
        )
        .reset_index(drop=True)
    )

    decile_data["decile"] = pd.qcut(
        np.arange(len(decile_data)),
        q=number_of_deciles,
        labels=range(1, number_of_deciles + 1)
    ).astype(int)

    decile_results = []

    for decile, group in decile_data.groupby(
        "decile",
        sort=True
    ):
        treated_group = group[
            group["treatment"] == 1
        ]

        control_group = group[
            group["treatment"] == 0
        ]

        treated_visit_rate = (
            treated_group["y_true"].mean()
        )

        control_visit_rate = (
            control_group["y_true"].mean()
        )

        decile_results.append({
            "decile": int(decile),
            "customers": int(len(group)),
            "mean_predicted_uplift": (
                group["predicted_uplift"].mean()
            ),
            "treated_customers": int(len(treated_group)),
            "control_customers": int(len(control_group)),
            "treated_visit_rate": treated_visit_rate,
            "control_visit_rate": control_visit_rate,
            "observed_uplift": (
                treated_visit_rate
                - control_visit_rate
            ),
        })

    return pd.DataFrame(decile_results)

### Calculate model uplift by decile

This cell applies the same decile-ranking procedure to the logistic-regression and XGBoost uplift predictions.

In [25]:
lgr_decile_results = calculate_uplift_by_decile(
    y_true=y_test,
    treatment=treatment_test,
    predicted_uplift=lgr_predicted_uplift
)

xgb_decile_results = calculate_uplift_by_decile(
    y_true=y_test,
    treatment=treatment_test,
    predicted_uplift=xgb_predicted_uplift
)

rf_decile_results = calculate_uplift_by_decile(
    y_true=y_test,
    treatment=treatment_test,
    predicted_uplift=rf_predicted_uplift
)


print("Logistic-regression deciles:")
display(lgr_decile_results)

print("XGBoost deciles:")
display(xgb_decile_results)

print("Random Forest deciles:")
display(rf_decile_results)

Logistic-regression deciles:
XGBoost deciles:
Random Forest deciles:


,decile,customers,mean_predicted_uplift,treated_customers,control_customers,treated_visit_rate,control_visit_rate,observed_uplift
0,1,1600,0.096406,1032,568,0.241279,0.130282,0.110997
1,2,1600,0.084619,1039,561,0.220404,0.135472,0.084932
2,3,1600,0.079166,1106,494,0.169982,0.117409,0.052573
3,4,1600,0.072360,1076,524,0.169145,0.095420,0.073725
4,5,1600,0.066352,1080,520,0.160185,0.092308,0.067877
5,6,1600,0.053801,1047,553,0.132760,0.079566,0.053194
6,7,1600,0.046161,1083,517,0.135734,0.096712,0.039022
7,8,1600,0.042386,1065,535,0.116432,0.072897,0.043535
8,9,1600,0.037158,1081,519,0.136910,0.067437,0.069473
9,10,1600,0.020562,1065,535,0.192488,0.170093,0.022395


,decile,customers,mean_predicted_uplift,treated_customers,control_customers,treated_visit_rate,control_visit_rate,observed_uplift
0,1,1600,0.111697,1047,553,0.222541,0.144665,0.077875
1,2,1600,0.088448,1065,535,0.209390,0.114019,0.095371
2,3,1600,0.078284,1069,531,0.173059,0.103578,0.069481
3,4,1600,0.070279,1060,540,0.144340,0.088889,0.055451
4,5,1600,0.062832,1093,507,0.173833,0.096647,0.077187
5,6,1600,0.053526,1059,541,0.162417,0.127542,0.034876
6,7,1600,0.046009,1051,549,0.147479,0.098361,0.049118
7,8,1600,0.040001,1075,525,0.147907,0.100952,0.046955
8,9,1600,0.032815,1082,518,0.130314,0.088803,0.041511
9,10,1600,0.015974,1073,527,0.161230,0.094877,0.066354


,decile,customers,mean_predicted_uplift,treated_customers,control_customers,treated_visit_rate,control_visit_rate,observed_uplift
0,1,1600,0.109206,1052,548,0.223384,0.164234,0.059150
1,2,1600,0.089670,1069,531,0.186155,0.099812,0.086344
2,3,1600,0.080126,1068,532,0.177903,0.084586,0.093316
3,4,1600,0.072019,1050,550,0.169524,0.105455,0.064069
4,5,1600,0.063713,1076,524,0.178439,0.101145,0.077294
5,6,1600,0.054799,1068,532,0.157303,0.127820,0.029484
6,7,1600,0.046146,1045,555,0.151196,0.095495,0.055701
7,8,1600,0.038621,1081,519,0.138760,0.084778,0.053982
8,9,1600,0.030129,1110,490,0.129730,0.091837,0.037893
9,10,1600,0.014378,1055,545,0.161137,0.102752,0.058385


### Plot observed uplift by predicted-uplift decile

This chart tests whether customers ranked in the highest predicted-uplift deciles demonstrate stronger observed treatment-control lift. 1= Highest predicted, 10= lowest predicted

In [26]:
deciles = np.arange(1, 11)
bar_width = 0.25

plt.figure(figsize=(12, 6))

plt.bar(
    deciles - bar_width,
    lgr_decile_results["observed_uplift"],
    width=bar_width,
    label="Logistic T-learner"
)

plt.bar(
    deciles,
    xgb_decile_results["observed_uplift"],
    width=bar_width,
    label="XGBoost T-learner"
)

plt.bar(
    deciles + bar_width,
    rf_decile_results["observed_uplift"],
    width=bar_width,
    label="Random Forest T-learner"
)

plt.axhline(
    observed_test_ate,
    color="black",
    linestyle="--",
    label=f"Random targeting expectation = {observed_test_ate:.3f}"
)

plt.xlabel("Predicted-Uplift Decile (1 = Highest)")
plt.ylabel("Observed Treatment-Control Lift")
plt.title("Observed Uplift by Predicted-Uplift Decile")
plt.xticks(deciles)
plt.legend()
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "t_learner_uplift_by_decile.png",
    dpi=150
)

plt.show()

### Calculate Qini and AUUC scores

In [27]:
causalml_evaluation_df = pd.DataFrame({
    "y": y_test.to_numpy(),
    "w": treatment_test.to_numpy(),
    "Logistic T-learner": np.asarray(lgr_predicted_uplift),
    "XGBoost T-learner": np.asarray(xgb_predicted_uplift),
    "Random Forest T-learner": np.asarray(rf_predicted_uplift),
})

causalml_qini_scores = qini_score(
    causalml_evaluation_df,
    outcome_col="y",
    treatment_col="w",
    normalize=True,
)

causalml_auuc_scores = auuc_score(
    causalml_evaluation_df,
    outcome_col="y",
    treatment_col="w",
    normalize=True,
)

lgr_qini_score = float(causalml_qini_scores["Logistic T-learner"])
xgb_qini_score = float(causalml_qini_scores["XGBoost T-learner"])
rf_qini_score = float(causalml_qini_scores["Random Forest T-learner"])

lgr_auuc_score = float(causalml_auuc_scores["Logistic T-learner"])
xgb_auuc_score = float(causalml_auuc_scores["XGBoost T-learner"])
rf_auuc_score = float(causalml_auuc_scores["Random Forest T-learner"])

uplift_evaluation = pd.DataFrame({
    "Model": [
        "Logistic T-learner",
        "XGBoost T-learner",
        "Random Forest T-learner",
    ],
    "Qini score": [
        lgr_qini_score,
        xgb_qini_score,
        rf_qini_score,
    ],
    "AUUC": [
        lgr_auuc_score,
        xgb_auuc_score,
        rf_auuc_score,
    ],
})

uplift_evaluation


,Model,Qini score,AUUC
0,Logistic T-learner,0.087694,0.590534
1,XGBoost T-learner,0.052379,0.554409
2,Random Forest T-learner,0.044387,0.545956


### Qini plot

In [28]:
t_learner_tau = {
    "Logistic T-learner": lgr_predicted_uplift,
    "XGBoost T-learner": xgb_predicted_uplift,
    "Random Forest T-learner": rf_predicted_uplift,
}
fig_qini = plot_qini_curve(
    y_test, treatment_test, t_learner_tau, normalize=True, figsize=(10, 6)
)
save_fig(fig_qini, "t_learner_qini_curves.png")

Figure(1000x600)


PosixPath('/workspace/scratch/61d1d4e14f7f/hillstrom-email-uplift-modeling/outputs/figures/t_learner_qini_curves.png')

### Cumulative-gain/AUUC plot

In [29]:
fig_gain = plot_cumulative_gain_curve(
    y_test, treatment_test, t_learner_tau, normalize=True, figsize=(10, 6)
)
save_fig(fig_gain, "t_learner_uplift_curves.png")

Figure(1000x600)


PosixPath('/workspace/scratch/61d1d4e14f7f/hillstrom-email-uplift-modeling/outputs/figures/t_learner_uplift_curves.png')

### Combine decile results

This cell combines the logistic-regression and XGBoost decile tables into one model-comparison output.

In [30]:
lgr_decile_output = lgr_decile_results.copy()
lgr_decile_output.insert(
    0,
    "model",
    "Logistic T-learner"
)

xgb_decile_output = xgb_decile_results.copy()
xgb_decile_output.insert(
    0,
    "model",
    "XGBoost T-learner"
)

rf_decile_output = rf_decile_results.copy()
rf_decile_output.insert(
    0,
    "model",
    "Random Forest T-learner"
)

decile_comparison = pd.concat(
    [
        lgr_decile_output,
        xgb_decile_output,
        rf_decile_output,
    ],
    ignore_index=True
)

display(decile_comparison)

,model,decile,customers,mean_predicted_uplift,treated_customers,control_customers,treated_visit_rate,control_visit_rate,observed_uplift
0,Logistic T-learner,1,1600,0.096406,1032,568,0.241279,0.130282,0.110997
1,Logistic T-learner,2,1600,0.084619,1039,561,0.220404,0.135472,0.084932
2,Logistic T-learner,3,1600,0.079166,1106,494,0.169982,0.117409,0.052573
3,Logistic T-learner,4,1600,0.072360,1076,524,0.169145,0.095420,0.073725
4,Logistic T-learner,5,1600,0.066352,1080,520,0.160185,0.092308,0.067877
5,Logistic T-learner,6,1600,0.053801,1047,553,0.132760,0.079566,0.053194
6,Logistic T-learner,7,1600,0.046161,1083,517,0.135734,0.096712,0.039022
7,Logistic T-learner,8,1600,0.042386,1065,535,0.116432,0.072897,0.043535
8,Logistic T-learner,9,1600,0.037158,1081,519,0.136910,0.067437,0.069473
9,Logistic T-learner,10,1600,0.020562,1065,535,0.192488,0.170093,0.022395


### Save prediction and evaluation tables

This cell saves the customer predictions, factual metrics, uplift metrics, and decile results for later project analysis.

In [31]:
predictions_path = (
    PREDICTIONS_DIR
    / "t_learner_predictions.csv"
)

factual_results_path = (
    TABLES_DIR
    / "t_learner_factual_performance.csv"
)

uplift_results_path = (
    TABLES_DIR
    / "t_learner_uplift_performance.csv"
)

decile_results_path = (
    TABLES_DIR
    / "t_learner_decile_results.csv"
)

model_comparison_df.to_csv(
    predictions_path,
    index=False
)

oof_predictions_path = PREDICTIONS_DIR / "t_learner_oof_predictions.csv"
oof_predictions_df = pd.DataFrame({
    "train_row_id": np.arange(len(y_train)),
    "y_true": y_train.to_numpy(),
    "treatment": treatment_train.to_numpy(),
    "lgr_predicted_uplift": t_oof["T-Learner (LR)"],
    "xgb_predicted_uplift": t_oof["T-Learner (XGB)"],
    "rf_predicted_uplift": t_oof["T-Learner (RF)"],
})
oof_predictions_df.to_csv(oof_predictions_path, index=False)

factual_comparison.to_csv(
    factual_results_path,
    index=False
)

uplift_evaluation.to_csv(
    uplift_results_path,
    index=False
)

decile_comparison.to_csv(
    decile_results_path,
    index=False
)

print("Test predictions saved to:", predictions_path)
print("OOF predictions saved to:", oof_predictions_path)
print("Factual results saved to:", factual_results_path)
print("Uplift results saved to:", uplift_results_path)
print("Decile results saved to:", decile_results_path)

Test predictions saved to: /workspace/scratch/61d1d4e14f7f/hillstrom-email-uplift-modeling/outputs/predictions/t_learner_predictions.csv
OOF predictions saved to: /workspace/scratch/61d1d4e14f7f/hillstrom-email-uplift-modeling/outputs/predictions/t_learner_oof_predictions.csv
Factual results saved to: /workspace/scratch/61d1d4e14f7f/hillstrom-email-uplift-modeling/outputs/tables/t_learner_factual_performance.csv
Uplift results saved to: /workspace/scratch/61d1d4e14f7f/hillstrom-email-uplift-modeling/outputs/tables/t_learner_uplift_performance.csv
Decile results saved to: /workspace/scratch/61d1d4e14f7f/hillstrom-email-uplift-modeling/outputs/tables/t_learner_decile_results.csv


### Save model results and metadata

This cell saves the main model results in a structured JSON file for reproducibility.

In [32]:
def summarize_cv(search):
    return {
        "best_parameters": search.best_params_,
        "mean_validation_log_loss": float(
            -search.best_score_
        ),
    }


results = {
    "metadata": {
        "model_type": "binary_treatment_t_learner",
        "outcome": PRIMARY_OUTCOME,
        "treatment_definition": "Any email versus no email",
        "test_size": int(len(y_test)),
        "observed_test_ate": float(observed_test_ate),
        "cross_validation_folds": cv_strategy.n_splits,
        "cross_validation_scoring": "log_loss",
    },
    "lgr_t_learner": {
        "base_learner": "logistic_regression",
        "base_parameters": lgr_parameters,
        "cross_validation": {
            "treated_model": summarize_cv(lgr_treated_cv),
            "control_model": summarize_cv(lgr_control_cv),
        },
        "average_predicted_uplift": float(
            lgr_average_predicted_uplift
        ),
        "minimum_predicted_uplift": float(
            lgr_minimum_predicted_uplift
        ),
        "maximum_predicted_uplift": float(
            lgr_maximum_predicted_uplift
        ),
        "treated_model_auc": float(lgr_treated_auc),
        "control_model_auc": float(lgr_control_auc),
        "treated_model_brier": float(lgr_treated_brier),
        "control_model_brier": float(lgr_control_brier),
        "qini_score": float(lgr_qini_score),
        "auuc_score": float(lgr_auuc_score),
    },
    "xgb_t_learner": {
        "base_learner": "xgboost",
        "base_parameters": xgb_parameters,
        "cross_validation": {
            "treated_model": summarize_cv(xgb_treated_cv),
            "control_model": summarize_cv(xgb_control_cv),
        },
        "average_predicted_uplift": float(
            xgb_average_predicted_uplift
        ),
        "minimum_predicted_uplift": float(
            xgb_minimum_predicted_uplift
        ),
        "maximum_predicted_uplift": float(
            xgb_maximum_predicted_uplift
        ),
        "treated_model_auc": float(xgb_treated_auc),
        "control_model_auc": float(xgb_control_auc),
        "treated_model_brier": float(xgb_treated_brier),
        "control_model_brier": float(xgb_control_brier),
        "qini_score": float(xgb_qini_score),
        "auuc_score": float(xgb_auuc_score),
    },
    "rf_t_learner": {
        "base_learner": "random_forest",
        "base_parameters": rf_parameters,
        "cross_validation": {
            "treated_model": summarize_cv(rf_treated_cv),
            "control_model": summarize_cv(rf_control_cv),
        },
        "average_predicted_uplift": float(
            rf_average_predicted_uplift
        ),
        "minimum_predicted_uplift": float(
            rf_minimum_predicted_uplift
        ),
        "maximum_predicted_uplift": float(
            rf_maximum_predicted_uplift
        ),
        "treated_model_auc": float(rf_treated_auc),
        "control_model_auc": float(rf_control_auc),
        "treated_model_brier": float(rf_treated_brier),
        "control_model_brier": float(rf_control_brier),
        "qini_score": float(rf_qini_score),
        "auuc_score": float(rf_auuc_score),
    },
}


results["validation_uplift_metrics"] = {
    "method": f"{N_OOF_FOLDS}-fold out-of-fold predictions on X_train",
    "scores": t_validation_comparison.to_dict(orient="index"),
}

results_path = (
    MODEL_RESULTS_DIR
    / "t_learner_results.json"
)

with open(results_path, "w") as file:
    json.dump(
        results,
        file,
        indent=4
    )

print("Model results saved to:", results_path)

Model results saved to: /workspace/scratch/61d1d4e14f7f/hillstrom-email-uplift-modeling/outputs/model_results/t_learner_results.json


### Summary

This notebook compared logistic-regression, XGBoost, and Random Forest T-learners using factual outcome-model performance, predicted-uplift distributions, uplift-by-decile results, Qini curves, and normalized AUUC scores. The preferred targeting model should demonstrate stronger observed lift in its highest-ranked deciles and higher positive Qini and AUUC scores than random targeting.